# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#%pip install moviepy==1.0.3
#%pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_15432\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [2]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 0 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (24 articulos)...
  Procesando: La jueza de la dana cita como testigo a Mazón...
    -> Texto completo (2723 chars)
  Procesando: La jefa de prensa de Mazón justificó ante la jueza de l...
    -> Texto completo (9137 chars)
  Procesando: El Poder Judicial ordena a su autoridad disciplinaria q...
    -> Texto completo (6359 chars)
  Procesando: Israel confirma su responsabilidad en bombardeos contra...
    -> Texto completo (2277 chars)
  Procesando: Guerra en Irán | Directo: Israel reivindica bombardeos ...
    -> Texto completo (2435 chars)
  Procesando: Junts apoyará el decreto anticrisis si el PSOE incluye ...
  

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [3]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ElDiario
Título:  La jueza de la dana cita como testigo a Mazón
Origen:  completo
Chars:   2723
Preview: La jueza de la dana cita como testigo a Mazón
La jueza de la dana ha citado a declarar como testigo al expresident de la Generalitat Carlos Mazón.
La magistrada ha acordado la citación en un auto dict...

Artículo 2
Fuente:  ElDiario
Título:  La jefa de prensa de Mazón justificó ante la jueza de la dana las mentiras sobre sus pasos: "Nunca con intención de engañar"
Origen:  completo
Chars:   9137
Preview: La jefa de prensa de Mazón justificó ante la jueza de la dana las mentiras sobre sus pasos: “Nunca con intención de engañar”
LEER ESTE TEXTO EN CATALÁN
La jefa de prensa de Carlos Mazón dijo ante la j...

Artículo 3
Fuente:  ElDiario
Título:  El Poder Judicial ordena a su autoridad disciplinaria que siga investigando las quejas de Bolaños contra el juez Peinado
Origen:  completo
Chars:   6359
Preview: El Poder Judicial ordena a su autoridad disciplinaria que si

## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [ ]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
#GEMINI_API_KEY = "..."

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 24 ARTÍCULOS

-> Agrupando artículos por temas...
   Error agrupando temas: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 8.698697232s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 8
}
]
Error agrupando, aborta

TypeError: cannot unpack non-iterable NoneType object

## Paso 3: Síntesis de voz (T2S)

In [5]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       3264
  Audio:       audios/audio_2026-03-24_18-30-10.mp3
Audio generado! (1.12 MB)
  Generando subtítulos con Whisper...


c:\Users\fcalv\anaconda3\envs\MCD_25_26\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Subtítulos generados: audios/subtitulos_2026-03-24_18-30-10.vtt


In [6]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 7734 chars
WEBVTT

00:00:00.000 --> 00:00:01.199
Muy buenas tardes y

00:00:01.199 --> 00:00:02.480
bienvenidos a nuestro repaso

00:00:02.480 --> 00:00:03.359
por la actualidad.

00:00:04.280 --> 00:00:05.719
Comenzamos con una noticia

00:00:05.719 --> 00:00:06.879
destacada en el ámbito

00:00:06.879 --> 00


## Paso 4: Generación del video resumen 

In [7]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [ ]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')
#PEXELS_API_KEY = "..."

ruta_video = generar_video(
    ruta_audio, ruta_subtitulos, resumenes,
    PEXELS_API_KEY,
    modelo_ia=mi_modelo_gemini  # para mejorar las queries de imagen
)

TypeError: deprecated_version_of() got an unexpected keyword argument 'oldname'. Did you mean 'old_name'?